# 03. Forecast & Valuation Bridge

This notebook converts operating evidence into a **first-pass independent forecast** for PER / RIM / DCF. Forecasts are scenarios, not company guidance or analyst consensus.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path('../data/processed')
hist = pd.read_csv(DATA / 'dcf_operating_inputs_2023_2025.csv')
h1 = pd.read_csv(DATA / 'h1_2026_dcf_snapshot.csv')
fcst = pd.read_csv(DATA / 'forecast_scenarios_2026_2030.csv')
hist, h1.head(), fcst.head()


## 1. 2026H1 anchor

2026H1 changes the DCF interpretation materially:
- revenue: KRW 1,484.7bn
- operating profit: KRW 353.3bn (23.8% margin)
- net income: KRW 282.1bn
- PPE acquisition: KRW 103.3bn vs KRW 324.9bn in 2025H1
- simplified core NWC: ~KRW 359.8bn vs KRW 300.6bn at 2025 year-end

Therefore, 2025's 19.1% CAPEX/sales ratio should **not** be extrapolated as steady-state CAPEX.


In [ ]:
h1.set_index('metric')[['2025_12_31_krw_bn','2026_06_30_krw_bn']]


In [ ]:
h1_revenue = 1484.714694859
h1_op = 353.313697185
h1_ni = 282.054263527
print(f'2026H1 operating margin: {h1_op/h1_revenue:.1%}')
print(f'2026H1 net margin: {h1_ni/h1_revenue:.1%}')


## 2. Scenario logic

### Base
- 2026 revenue KRW 3.10tn: implies H2 revenue of ~KRW 1.62tn, only modest sequential expansion from Q2.
- 2027 growth 20%: first year of Jiaxing China plant commissioning plus continued U.S./Europe distribution expansion.
- growth fades to 9% by 2030 instead of extending recent hyper-growth indefinitely.
- operating margin remains around 23.5-24.0%, anchored to 2026H1 rather than assuming continuous margin expansion.
- growth CAPEX falls sharply after the current expansion cycle; maintenance CAPEX rises with the asset base.
- core NWC/sales gradually normalizes from ~12% as distribution and inventory scale mature.

### Bull
Faster overseas sell-through, successful China localization, sustained mix/pricing and further capacity investments.

### Bear
Slower overseas growth, margin normalization, weaker operating leverage and less incremental growth CAPEX.


In [ ]:
summary_cols = ['scenario','year','revenue_krw_bn','revenue_growth','operating_margin',
                'net_income_krw_bn','eps_krw','capex_krw_bn','fcff_krw_bn']
fcst[summary_cols]


In [ ]:
fig, ax = plt.subplots(figsize=(8,4.5))
for scenario, g in fcst.groupby('scenario'):
    ax.plot(g['year'], g['revenue_krw_bn'], marker='o', label=scenario)
ax.set_title('Samyang Foods Revenue Scenarios')
ax.set_ylabel('KRW bn')
ax.legend()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8,4.5))
for scenario, g in fcst.groupby('scenario'):
    ax.plot(g['year'], g['fcff_krw_bn'], marker='o', label=scenario)
ax.set_title('FCFF Scenarios Before WACC / Terminal Value')
ax.set_ylabel('KRW bn')
ax.legend()
plt.show()


## 3. 2026 reasonableness check

2025 full-year revenue was KRW 2,351.8bn and 2025H1 revenue was KRW 1,082.1bn. The Base 2026 forecast of KRW 3,100bn therefore implies H2 2026 revenue of about KRW 1,615bn, roughly 27% above 2025H2.

This is deliberately slower than 2026H1's 37% YoY growth and therefore assumes some growth normalization despite new capacity.


In [ ]:
rev_2025 = 2351.785
h1_2025 = 1082.090816109
h2_2025 = rev_2025 - h1_2025
for scenario, rev_2026 in [('Base',3100),('Bull',3250),('Bear',2950)]:
    h2_2026 = rev_2026 - h1_revenue
    print(scenario, 'H2 2026 implied:', round(h2_2026,1), 'KRW bn | YoY:', f'{h2_2026/h2_2025-1:.1%}')


## 4. Bridge to valuation

### PER
Use `net_income_krw_bn` and `eps_krw`, then build a peer set and justify the forward multiple / premium or discount.

### RIM
Next step: forecast beginning equity, ROE and dividends/retention. 2026H1 attributable equity provides a fresh anchor.

### DCF
`fcff_krw_bn` is ready for discounting. **Do not pick WACC just to obtain a desired target price.** Estimate risk-free rate, beta, ERP and debt cost separately, then run WACC × terminal-growth sensitivity.

## Remaining refinements
1. split consolidated overseas revenue into U.S. / China / Europe / Other through 2030,
2. model raw materials / FX / marketing effects on margin,
3. reconcile D&A including right-of-use and intangible amortization,
4. reconcile full operating NWC beyond the simplified AR + inventory - AP proxy,
5. estimate WACC and net debt for DCF,
6. create peer PER set and RIM equity roll-forward.
